# Cell 1 — Notebook Settings
Set the run mode, output file, debug options, and timing options.

In [ ]:
# ------------------------------------------------------------
# NOTEBOOK SETTINGS
# ------------------------------------------------------------

# TABLE_SOURCE_MODE options:
#
# "json" = use tables_to_compare.json
# "sql"  = discover tables automatically
#

# TABLE_SOURCE_MODE = "json"
TABLE_SOURCE_MODE = "sql"


# ------------------------------------------------------------
# OUTPUT SETTINGS
# ------------------------------------------------------------

EXPORT_TO_EXCEL = True

OUTPUT_FILE = "../output/schema_compare_results.xlsx"


# ------------------------------------------------------------
# DEBUG SETTINGS
# ------------------------------------------------------------

SHOW_FIRST_N_TABLES = 20

# ------------------------------------------------------------
# TIMER SETTINGS
# ------------------------------------------------------------

SHOW_TIMING = True

# Cell 2 — Imports and Environment Check
Import required Python packages and confirm the notebook is using the correct Python interpreter.

In [ ]:
import sys
print(sys.executable)

from dotenv import load_dotenv
import pandas as pd
import pyodbc
import os

from pathlib import Path 
import json

print("Imports successful")

# Cell 3 — Start Timer
Record the notebook start time so the full run duration can be calculated at the end.

In [ ]:
from datetime import datetime
import time

start_datetime = datetime.now()
start_time = time.time()

if SHOW_TIMING:

    print("START TIME:", start_datetime.strftime("%Y-%m-%d %H:%M:%S"))

# Cell 4 — Load Environment Variables
Load private settings from the local `.env` file, such as the Azure SQL Server name.

In [ ]:
load_dotenv()


SERVER = os.getenv("SQL_SERVER")

print("SERVER:", SERVER)


# Cell 5 — Create SQL Connection Helper
Create a reusable function for connecting to each Azure SQL database.

In [ ]:

# ------------------------------------------------------------
# Load databases from JSON
# ------------------------------------------------------------

with open("../config/databases.json", "r") as f:
    DATABASES = json.load(f)

print("DATABASES")
print(DATABASES)


# ------------------------------------------------------------
# Load tables from JSON mode
# ------------------------------------------------------------

if TABLE_SOURCE_MODE == "json":

    with open("../config/tables_to_compare.json", "r") as f:
        table_config = json.load(f)

    TABLES_TO_COMPARE = [
        (x["schema"], x["table"])
        for x in table_config
    ]


# ------------------------------------------------------------
# Load tables from SQL discovery mode
# ------------------------------------------------------------

elif TABLE_SOURCE_MODE == "sql":

    def get_base_tables(database: str):

        sql = """
        SELECT
             s.name AS schema_name
            ,t.name AS table_name
        FROM sys.tables t
        JOIN sys.schemas s
            ON t.schema_id = s.schema_id
        WHERE t.is_ms_shipped = 0
          AND s.name NOT IN ('sys', 'INFORMATION_SCHEMA')
        ORDER BY
             s.name
            ,t.name;
        """

        with get_conn(database) as conn:
            return pd.read_sql(sql, conn)

    table_rows = []

    for database in DATABASES:

        df = get_base_tables(database)
        df["database_name"] = database

        table_rows.append(df)

        print(f"Loaded table list from {database}: {len(df)} tables")

    all_tables_found = pd.concat(table_rows, ignore_index=True)

    TABLES_TO_COMPARE = (
        all_tables_found[
            ["schema_name", "table_name"]
        ]
        .drop_duplicates()
        .sort_values(["schema_name", "table_name"])
        .apply(
            lambda row: (row["schema_name"], row["table_name"]),
            axis=1
        )
        .tolist()
    )


# ------------------------------------------------------------
# Invalid mode check
# ------------------------------------------------------------

else:

    raise ValueError("TABLE_SOURCE_MODE must be either 'json' or 'sql'")


# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------

print("\nTABLES_TO_COMPARE")
print(f"Total tables: {len(TABLES_TO_COMPARE)}")

for table in TABLES_TO_COMPARE[:SHOW_FIRST_N_TABLES]:
    print(table)

if len(TABLES_TO_COMPARE) > SHOW_FIRST_N_TABLES:

    print(
        f"... showing first "
        f"{SHOW_FIRST_N_TABLES} "
        f"of {len(TABLES_TO_COMPARE)} tables"
    )

# Cell 6 — Load Database and Table Configuration
Load the database list and either load selected tables from JSON or discover all tables from SQL.

In [ ]:
def get_conn(database: str):

    conn_str = (
        "Driver={ODBC Driver 18 for SQL Server};"
        f"Server={SERVER};"
        f"Database={database};"
        "Authentication=ActiveDirectoryInteractive;"
        "Encrypt=yes;"
        "TrustServerCertificate=no;"
    )

    return pyodbc.connect(conn_str)

# Cell 7 — Test Database Connection
Connect to the first configured database and confirm the login and server settings are working.

In [ ]:
test_db = DATABASES[0]

with get_conn(test_db) as conn:

    cursor = conn.cursor()

    cursor.execute("SELECT DB_NAME()")

    print(cursor.fetchone()[0])

# Cell 8 — Define Schema Reader
Create a function that reads column names, data types, lengths, nullability, and identity settings.

In [ ]:
def get_table_schema(database, schema_name, table_name):

    sql = """
    SELECT
         DB_NAME() AS database_name
        ,s.name AS schema_name
        ,t.name AS table_name
        ,c.column_id
        ,c.name AS column_name
        ,ty.name AS data_type
        ,c.max_length
        ,c.precision
        ,c.scale
        ,c.is_nullable
        ,c.is_identity
    FROM sys.tables t
    JOIN sys.schemas s
        ON t.schema_id = s.schema_id
    JOIN sys.columns c
        ON t.object_id = c.object_id
    JOIN sys.types ty
        ON c.user_type_id = ty.user_type_id
    WHERE s.name = ?
      AND t.name = ?
    ORDER BY c.column_id
    """

    with get_conn(database) as conn:

        return pd.read_sql(
            sql,
            conn,
            params=[schema_name, table_name]
        )

# Cell 9 — Define Primary Key Reader
Create a function that reads primary key names and primary key columns for each table.

In [ ]:
def get_primary_keys(database, schema_name, table_name):

    sql = """
    SELECT
         DB_NAME() AS database_name
        ,s.name AS schema_name
        ,t.name AS table_name
        ,kc.name AS pk_name
        ,c.name AS column_name
        ,ic.key_ordinal
    FROM sys.key_constraints kc
    JOIN sys.tables t
        ON kc.parent_object_id = t.object_id
    JOIN sys.schemas s
        ON t.schema_id = s.schema_id
    JOIN sys.index_columns ic
        ON kc.parent_object_id = ic.object_id
       AND kc.unique_index_id = ic.index_id
    JOIN sys.columns c
        ON ic.object_id = c.object_id
       AND ic.column_id = c.column_id
    WHERE kc.type = 'PK'
      AND s.name = ?
      AND t.name = ?
    ORDER BY ic.key_ordinal
    """

    with get_conn(database) as conn:

        return pd.read_sql(
            sql,
            conn,
            params=[schema_name, table_name]
        )

# Cell 10 — Collect Metadata From All Databases
Loop through each database and table, collecting schema and primary key metadata.

In [ ]:
schema_results = []
pk_results = []

for database in DATABASES:

    for schema_name, table_name in TABLES_TO_COMPARE:

        try:

            schema_df = get_table_schema(
                database,
                schema_name,
                table_name
            )

            pk_df = get_primary_keys(
                database,
                schema_name,
                table_name
            )

            schema_results.append(schema_df)
            pk_results.append(pk_df)

            print(f"SUCCESS: {database}.{schema_name}.{table_name}")

        except Exception as e:

            print(f"FAILED: {database}.{schema_name}.{table_name}")
            print(e)

all_schema = pd.concat(
    schema_results,
    ignore_index=True
)

all_pks = pd.concat(
    pk_results,
    ignore_index=True
)

# Cell 11 — View Raw Schema Results
Display the full collected schema results before comparison.

In [ ]:
display(all_schema)

# Cell 12 — Compare Column Presence
Show which columns exist or are missing across the selected databases.

In [ ]:
column_presence = (
    all_schema
    .pivot_table(
        index=[
            "schema_name",
            "table_name",
            "column_name"
        ],
        columns="database_name",
        values="data_type",
        aggfunc="first"
    )
)

display(column_presence)

# Cell 13 — Detect Schema Mismatches
Find columns where data type, length, precision, scale, nullability, or identity settings do not match.

In [ ]:
mismatches = (
    all_schema
    .groupby([
        "schema_name",
        "table_name",
        "column_name"
    ])
    .agg(
        data_type_count=("data_type", "nunique"),
        nullable_count=("is_nullable", "nunique"),
        identity_count=("is_identity", "nunique"),
        max_length_count=("max_length", "nunique"),
        precision_count=("precision", "nunique"),
        scale_count=("scale", "nunique")
    )
    .reset_index()
)

mismatches = mismatches[
    (mismatches["data_type_count"] > 1)
    |
    (mismatches["nullable_count"] > 1)
    |
    (mismatches["identity_count"] > 1)
    |
    (mismatches["max_length_count"] > 1)
    |
    (mismatches["precision_count"] > 1)
    |
    (mismatches["scale_count"] > 1)
]

display(mismatches)

In [ ]:
# ============================================================
# FIND MISSING COLUMNS BETWEEN DATABASES
# ============================================================

# normalize column names
all_schema["column_name_norm"] = (
    all_schema["column_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

results = []

# build master list of columns for each table
master_columns = (
    all_schema
    .groupby(["schema_name", "table_name"])["column_name_norm"]
    .apply(lambda x: sorted(set(x)))
    .reset_index(name="expected_columns")
)

for _, row in master_columns.iterrows():

    schema_name = row["schema_name"]
    table_name = row["table_name"]
    expected_columns = set(row["expected_columns"])

    table_rows = all_schema[
        (all_schema["schema_name"] == schema_name)
        & (all_schema["table_name"] == table_name)
    ]

    for database_name, db_group in table_rows.groupby("database_name"):

        actual_columns = set(db_group["column_name_norm"])

        missing_columns = expected_columns - actual_columns

        for missing_col in sorted(missing_columns):

            databases_that_have_it = sorted(
                table_rows[
                    table_rows["column_name_norm"] == missing_col
                ]["database_name"].unique()
            )

            results.append({
                "database_missing_column": database_name,
                "schema_name": schema_name,
                "table_name": table_name,
                "missing_column": missing_col,
                "databases_that_have_column": ", ".join(databases_that_have_it)
            })

missing_columns_df = pd.DataFrame(results)

display(missing_columns_df)

# Cell 14 — View Detailed Mismatch Records
Show the database-level details for each mismatched column.

# Cell 15 — Compare Primary Keys
Display primary key names and columns across the compared databases.

In [ ]:
mismatch_detail = all_schema.merge(

    mismatches[
        [
            "schema_name",
            "table_name",
            "column_name"
        ]
    ],

    on=[
        "schema_name",
        "table_name",
        "column_name"
    ],

    how="inner"
)

display(mismatch_detail)

In [ ]:
display(all_pks)

# Cell 16 — Export Results and End Timer
Export the comparison results to Excel and print the total runtime.

In [ ]:
if EXPORT_TO_EXCEL:

    with pd.ExcelWriter(
        OUTPUT_FILE,
        engine="openpyxl"
    ) as writer:

        all_schema.to_excel(
            writer,
            sheet_name="all_schema",
            index=False
        )

        all_pks.to_excel(
            writer,
            sheet_name="primary_keys",
            index=False
        )

        mismatches.to_excel(
            writer,
            sheet_name="mismatches",
            index=False
        )

        mismatch_detail.to_excel(
            writer,
            sheet_name="mismatch_detail",
            index=False
        )

    print(f"\nSaved: {OUTPUT_FILE}")


# ------------------------------------------------------------
# END TIMER
# ------------------------------------------------------------

end_datetime = datetime.now()
end_time = time.time()

elapsed_seconds = end_time - start_time

elapsed_minutes = elapsed_seconds / 60


if SHOW_TIMING:

    print("\n-----------------------------------")
    print("START TIME :", start_datetime.strftime("%Y-%m-%d %H:%M:%S"))
    print("END TIME   :", end_datetime.strftime("%Y-%m-%d %H:%M:%S"))
    print(f"TOTAL TIME : {elapsed_seconds:,.2f} seconds")
    print(f"TOTAL TIME : {elapsed_minutes:,.2f} minutes")
    print("-----------------------------------")